# ViSceT5: tiền huấn luyện TWC → VQA
Chạy tất cả các ô theo thứ tự. Giai đoạn 1 chỉ dùng train, không đánh giá validation/test. Giai đoạn 2 dùng VQA loss và validation. Mỗi giai đoạn có repo và checkpoint riêng; ngắt kết nối có thể chạy lại để tiếp tục.


In [ ]:
import os, subprocess
from pathlib import Path
BRANCH = "tmp"
repo = Path("/content/ViSceT5")
if not repo.exists():
    subprocess.run(["git", "clone", "-b", BRANCH, "https://github.com/Kussssssss/ViSceT5.git", str(repo)], check=True)
os.chdir(repo)
for command in (["git", "fetch", "origin"], ["git", "checkout", BRANCH],
                ["git", "pull", "origin", BRANCH], ["bash", "setup.sh"]):
    subprocess.run(command, check=True)
for config in ("configs/pretrain.yaml", "configs/twc_vqa.yaml"):
    if not Path(config).is_file():
        raise RuntimeError(f"Thiếu cấu hình hai giai đoạn: {config}")


In [ ]:
import os
from google.colab import userdata
from huggingface_hub import HfApi, get_token
try:
    token = userdata.get("HF_TOKEN") or get_token()
except Exception:
    token = get_token()
if not token:
    raise RuntimeError("Cần đặt HF_TOKEN trong Colab Secrets trước khi train")
os.environ["HF_TOKEN"] = token
user = HfApi(token=token).whoami()["name"]
# Đổi RUN_ID cho thí nghiệm mới; giữ nguyên để resume đúng thí nghiệm.
RUN_ID = "vitextvqa-two-stage-v1"
assert len(RUN_ID) < 65
PRETRAIN_REPO = f"{user}/viscet5-{RUN_ID}-pretrain"
VQA_REPO = f"{user}/viscet5-{RUN_ID}-vqa"
PRETRAIN_DIR = f"/content/ViSceT5/output/{RUN_ID}-pretrain"
VQA_DIR = f"/content/ViSceT5/output/{RUN_ID}-vqa"
print(PRETRAIN_REPO, VQA_REPO, sep="\n")

# Xóa các override cũ từ notebook trước để YAML của mỗi stage có hiệu lực.
import run_pipeline
import inspect, re
stage_overrides = set(re.findall(r'"([A-Z][A-Z0-9_]+)"\s*:', inspect.getsource(run_pipeline.training_arguments)))
stage_overrides.update({"TRAIN_CONFIG", "OUTPUT_DIR", "MOCK_TEST", "CLIP_LATE_ADAPTER", "BOOTSTRAP_PRETRAIN", "STAGE", "HF_REPO", "RESUME_FROM_HF"})
def stage_env(**settings):
    env = {key: value for key, value in os.environ.items() if key not in stage_overrides}
    env.update(settings, MOCK_TEST="false")
    return env


## 1. TWC pretrain
ViT5 encoder, CLIP và hai lượt ConsFormer; không có decoder, question hay answer. p=0.35, τ cố định 0.07.


In [ ]:
import subprocess
pre_env = stage_env(STAGE="pretrain", TRAIN_CONFIG="configs/pretrain.yaml", HF_REPO=PRETRAIN_REPO, OUTPUT_DIR=PRETRAIN_DIR, RESUME_FROM_HF="1")
subprocess.run(["python", "run_pipeline.py"], env=pre_env, check=True)


## 2. VQA downstream
Khởi tạo decoder từ ViT5 pretrained, chuyển trọng số chung từ giai đoạn 1; optimizer và global step mới. p=0.7, chỉ VQA loss. Resume riêng từ repo VQA nếu đã có checkpoint.


In [ ]:
vqa_env = stage_env(STAGE="finetune", TRAIN_CONFIG="configs/twc_vqa.yaml", HF_REPO=VQA_REPO, OUTPUT_DIR=VQA_DIR, BOOTSTRAP_PRETRAIN=PRETRAIN_DIR, RESUME_FROM_HF="1")
subprocess.run(["python", "run_pipeline.py"], env=vqa_env, check=True)
